# MOMA-AC trên Kaggle: GPI-LS, GPI-PD và baseline random

Hướng dẫn đầy đủ nằm ở `kaggle/README.md`. Tóm tắt:

1. Gắn Dataset chứa code vào notebook, bật Internet.
2. Chạy thử: để `SMOKE_TEST = True`, Run All, chờ dòng `SMOKE TEST PASSED`.
3. Train thật: đổi `SMOKE_TEST = False`, rồi **Save Version → Save & Run All (Commit)**.
4. Chưa xong? Gắn output của version vừa rồi làm input, rồi Save Version lần nữa. Run tự chạy tiếp.

In [ ]:
# ── Cài đặt: chỉ cần sửa ở ô này ─────────────────────────────
SMOKE_TEST = True          # True: thử 1-3 phút. False: train thật
VARIANTS   = ["random", "gpi_ls", "gpi_pd"]
SEEDS      = [1]           # Lượt đầu 1 seed; sau đó [1, 2, 3], run đã xong không chạy lại
MAX_HOURS  = 10.5          # Giới hạn session 12 giờ: chừa chỗ cho eval, lưu file và upload output
DEVICE     = "cuda"        # Nhanh hơn CPU nhưng tốn quota GPU; xem bảng so sánh trong README
PARALLEL   = 3             # Số run song song, tự chia đều nếu máy có nhiều GPU

INPUT_ROOT = "/kaggle/input"
WORK_ROOT  = "/kaggle/working"
CODE_DIR   = "/kaggle/temp/moma_code"   # Không nằm trong output; numba cần ghi cache cạnh code

In [ ]:
# ── Tìm code trong Dataset và chép sang chỗ ghi được ─────────
import glob, os, shutil, subprocess, sys, zipfile

shutil.rmtree(CODE_DIR, ignore_errors=True)
zips    = glob.glob(os.path.join(INPUT_ROOT, "**", "moma_uav_code.zip"), recursive=True)
scripts = glob.glob(os.path.join(INPUT_ROOT, "**", "train_moma_ac.py"), recursive=True)
if scripts:
    shutil.copytree(os.path.dirname(scripts[0]), CODE_DIR)
    print("Code từ thư mục:", os.path.dirname(scripts[0]))
elif zips:
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall(CODE_DIR)
    print("Code từ zip:", zips[0])
else:
    raise FileNotFoundError("Không thấy code. Hãy gắn Dataset chứa moma_uav_code.zip vào notebook (Add Input).")

try:
    import numba
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numba"], check=True)

import torch, numba
print(f"torch {torch.__version__}, numba {numba.__version__}, CPUs={os.cpu_count()}, "
      f"GPU={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'không có'}")

In [ ]:
# ── Train ────────────────────────────────────────────────────
cmd = [sys.executable, os.path.join(CODE_DIR, "kaggle", "run_kaggle.py"),
       "--variants", *VARIANTS, "--seeds", *map(str, SEEDS),
       "--max_hours", str(MAX_HOURS), "--device", DEVICE,
       "--input_root", INPUT_ROOT, "--work_root", WORK_ROOT]
if PARALLEL:
    cmd += ["--parallel", str(PARALLEL)]
if SMOKE_TEST:
    cmd += ["--smoke", "--seeds", "1", "--max_hours", "0.5"]

subprocess.run(cmd)

In [ ]:
# ── Kết quả: bảng ở mốc eval cuối của mỗi variant ───────────
import pandas as pd

table = os.path.join(WORK_ROOT, "comparison.csv")
if os.path.exists(table):
    df = pd.read_csv(table)
    display(df[df["step"] == df.groupby("variant")["step"].transform("max")].T)
print(subprocess.run(["du", "-sh", WORK_ROOT], capture_output=True, text=True).stdout)